<a href="https://colab.research.google.com/github/nidhikulal11/Fish-Anomaly-Detection/blob/main/Copy_of_02_Phase2_CLAHE_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q opencv-python tqdm pandas numpy

In [ ]:
from pathlib import Path

input_dir = Path("/content/drive/MyDrive/FishProject/extracted_frames")

print("Exists:", input_dir.exists())

if input_dir.exists():
    folders = [f.name for f in input_dir.iterdir() if f.is_dir()]
    print("Folders:")
    for folder in folders:
        print("-", folder)

Exists: True
Folders:
- healthy
- unhealthy


In [ ]:

from pathlib import Path
import cv2
import os
import json
import logging
from tqdm import tqdm
import pandas as pd
import numpy as np
from datetime import datetime

ROOT = Path("/content/drive/MyDrive/FishProject")
INPUT_DIR = ROOT / "extracted_frames"
OUTPUT_DIR = ROOT / "clahe_frames"
REPORT_DIR = ROOT / "reports"
META_DIR = ROOT / "metadata"
COMPARE_DIR = ROOT / "comparison_images"
LOG_DIR = ROOT / "logs"

for d in [OUTPUT_DIR, REPORT_DIR, META_DIR, COMPARE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=str(LOG_DIR / "clahe_processing.log"),
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
summary=[]

def enhance_image(inp, out):
    img=cv2.imread(str(inp))
    if img is None:
        return False,"corrupted"

    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    l=clahe.apply(l)
    merged=cv2.merge((l,a,b))
    result=cv2.cvtColor(merged,cv2.COLOR_LAB2BGR)

    out.parent.mkdir(parents=True,exist_ok=True)
    cv2.imwrite(str(out),result)
    return True,result.shape

for path in tqdm(sorted(INPUT_DIR.rglob("*"))):
    if path.suffix.lower() not in [".jpg",".jpeg",".png",".bmp"]:
        continue

    rel=path.relative_to(INPUT_DIR)
    out=OUTPUT_DIR/rel

    if out.exists():
        continue

    ok,info=enhance_image(path,out)

    summary.append({
        "input":str(path),
        "output":str(out),
        "status":"success" if ok else "failed",
        "info":str(info)
    })

    if ok and len(summary)<=20:
        original=cv2.imread(str(path))
        enhanced=cv2.imread(str(out))
        comp=np.hstack([original,enhanced])
        cv2.imwrite(str(COMPARE_DIR/f"{path.stem}_comparison.jpg"),comp)

df=pd.DataFrame(summary)
csv_path=META_DIR/"clahe_summary.csv"
df.to_csv(csv_path,index=False)

stats={
    "processed":int((df.status=="success").sum()),
    "failed":int((df.status=="failed").sum()),
    "generated":len(list(OUTPUT_DIR.rglob("*.jpg"))),
    "time":datetime.now().isoformat()
}

with open(REPORT_DIR/"processing_statistics.json","w") as f:
    json.dump(stats,f,indent=4)

with open(REPORT_DIR/"clahe_processing_report.txt","w") as f:
    f.write("CLAHE PREPROCESSING REPORT\n")
    for k,v in stats.items():
        f.write(f"{k}: {v}\n")

print("Phase 2 completed.")
print(csv_path)


100%|██████████| 629/629 [04:05<00:00,  2.56it/s]

Phase 2 completed.
/content/drive/MyDrive/FishProject/metadata/clahe_summary.csv


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/FishProject")

print("CLAHE Frames:", (root/"clahe_frames").exists())
print("Comparison Images:", (root/"comparison_images").exists())
print("Metadata:", (root/"metadata").exists())
print("Reports:", (root/"reports").exists())
print("Logs:", (root/"logs").exists())

CLAHE Frames: True
Comparison Images: True
Metadata: True
Reports: True
Logs: True


deleted simplified phase 2

In [ ]:
import shutil
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/FishProject")

folders_to_delete = [
    ROOT / "clahe_frames",
    ROOT / "comparison_images",
    ROOT / "logs",
]

files_to_delete = [
    ROOT / "metadata" / "clahe_summary.csv",
    ROOT / "reports" / "clahe_processing_report.txt",
    ROOT / "reports" / "processing_statistics.json",
]

# Delete folders
for folder in folders_to_delete:
    if folder.exists():
        shutil.rmtree(folder)
        print(f"Deleted folder: {folder}")

# Delete files
for file in files_to_delete:
    if file.exists():
        file.unlink()
        print(f"Deleted file: {file}")

print("\nPhase 2 outputs cleaned successfully.")

Deleted folder: /content/drive/MyDrive/FishProject/clahe_frames
Deleted folder: /content/drive/MyDrive/FishProject/comparison_images
Deleted folder: /content/drive/MyDrive/FishProject/logs

Phase 2 outputs cleaned successfully.
